In [18]:
from ax.api.client import Client
from PlottingMethods import PlottingMethods_class
from ax.api.configs import RangeParameterConfig

In [19]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
client.get_next_trials(max_trials=1)

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning:

A not p.d., added jitter of 1.0e-08 to the diagonal



{302: {'s1': 0.33994697334815954,
  's2': 0.6654091000545805,
  'b1': 0.7637870480551815}}

In [20]:
Plotter = PlottingMethods_class()

In [21]:
class OptimisationSetup_class():
    def __init__(self):
        self.name = "Class of Plotting Methods"
        self.Parameters_lis = [
            RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
            RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
            RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]
OptimisationSetup_obj = OptimisationSetup_class()
Resolution_int = 10
TypeOfJob_str = "mean"
Plotter.InteractiveFunctionPlot(client,OptimisationSetup_obj,TypeOfJob_str,Resolution_int)

3D Interactive Plot


/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning:

A not p.d., added jitter of 1.0e-08 to the diagonal



In [2]:
import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from ax.utils.stats.model_fit_stats import MSE
from botorch.models import SingleTaskGP
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import plotly.express as px

from gpytorch.kernels import MaternKernel
from ax.models.torch.botorch_modular.kernels import DefaultRBFKernel, ScaleMaternKernel
from gpytorch.kernels.linear_kernel import LinearKernel
from gpytorch.kernels.rbf_kernel import RBFKernel

# Data Import

In [3]:
GrdSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-C1_PGCI-GrdSrch-[27]-P3O1/raw-data_2023-04-11_Stykke4.csv")
SobSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-C3_PGCI-SobSrch-[27]-P3O1/raw-data_2023-04-11_Stykke4.csv")
RndSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-A2_PGCI-RndSrch-[27]-P3O1/raw-data_2023-03-10_PtA2-PGCI-RndSrch-[27]-P3O1_Stykke-4.csv")

RndSrch_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)

BOpt_8SP_1It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B1_PGCI-BOpt-[8,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1]-P3O1/raw-data_2023-03-15_PtB1-PGCI-BOpt-[8,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1]-P3O1-Stykke-4.csv")
BOpt_8SP_2It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B3_PGCI-BOpt-[8,2,2,2,2,2,2,2,2,2,1]-P3O1/raw-data_2023-03-20_PtB3-PGCI-BOpt-[8,2,2,2,2,2,2,2,2,2,1]-P3O1-Stykke-4.csv")
BOpt_8SP_3It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B5_PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1/raw-data_2023-03-20_PtB5-PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1-Stykke-4.csv")
BOpt_16SP_1It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B2_PGCI-BOpt-[16,1,1,1,1,1,1,1,1,1,1,1]-P3O1/raw-data_2023-03-11_PtB2-PGCI-BOpt-[16,1,1,1,1,1,1,1,1,1,1,1]-P3O1-Stykke-4.csv")
BOpt_16SP_2It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B4_PGCI-BOpt-[16,2,2,2,2,1]-P3O1/raw-data_2023-03-20_PtB4-PGCI-BOpt-[16,2,2,2,2,1]-P3O1-Stykke-4.csv")
BOpt_16SP_3It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B6_PGCI-BOpt-[16,3,3,3,2]-P3O1/raw-data_2023-03-20_PtB6-PGCI-BOpt-[16,3,3,3,2]-P3O1-Stykke-4.csv")

BOpt_8SP_1It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
BOpt_8SP_2It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
BOpt_8SP_3It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
BOpt_16SP_1It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
BOpt_16SP_2It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
BOpt_16SP_3It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)

ax_BOpt_8SP_1It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-D1_PGCI-BOpt-9,27,1-S2B1O1/raw-data_2023-03-15_Stykke4.csv")
ax_BOpt_8SP_2It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-D2_PGCI-BOpt-9,27,2-S2B1O1/raw-data_2023-03-15_Stykke4.csv")
ax_BOpt_8SP_3It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-D3_PGCI-BOpt-9,27,3-S2B1O1/raw-data_2023-03-15_Stykke4.csv")
ax_BOpt_16SP_1It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-D4_PGCI-BOpt-17,27,1-S2B1O1/raw-data_2023-03-11_Stykke4.csv")
ax_BOpt_16SP_2It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-D5_PGCI-BOpt-17,27,2-S2B1O1/raw-data_2023-03-11_Stykke4.csv")
ax_BOpt_16SP_3It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-D6_PGCI-BOpt-17,27,3-S2B1O1/raw-data_2023-03-11_Stykke4.csv")

In [4]:
# df = pd.concat(objs=[GrdSrch_df,SobSrch_df,RndSrch_df,BOpt_8SP_1It_df,BOpt_8SP_2It_df,BOpt_8SP_3It_df,BOpt_16SP_1It_df,BOpt_16SP_2It_df,BOpt_16SP_3It_df,ax_BOpt_8SP_1It_df,ax_BOpt_8SP_2It_df,ax_BOpt_8SP_3It_df,ax_BOpt_16SP_1It_df,ax_BOpt_16SP_2It_df,ax_BOpt_16SP_3It_df])
# df = pd.concat(objs=[GrdSrch_df,SobSrch_df,RndSrch_df,BOpt_8SP_2It_df,BOpt_8SP_3It_df,BOpt_16SP_2It_df,BOpt_16SP_3It_df,ax_BOpt_8SP_2It_df,ax_BOpt_8SP_3It_df,ax_BOpt_16SP_2It_df,ax_BOpt_16SP_3It_df]) # nu 2.5 = mk5
df = pd.concat(objs=[GrdSrch_df,SobSrch_df,RndSrch_df,BOpt_8SP_2It_df,BOpt_8SP_3It_df,BOpt_16SP_2It_df,BOpt_16SP_3It_df,ax_BOpt_8SP_2It_df,ax_BOpt_8SP_3It_df,ax_BOpt_16SP_2It_df,ax_BOpt_16SP_3It_df]) # nu 1.5 = mk6
df.drop(columns=["mould_position","G_stoichiometry","CA_stoichiometry","IA_stoichiometry","StartPolymerMass_g","EndPolymerMass_pct"],inplace=True)
df['DeltaPolymerMass_pct']=df['DeltaPolymerMass_pct']*-1
df

,s1,s2,b1,DeltaPolymerMass_pct
0,0.000000,0.000000,0.000000,14.688580
1,0.000000,0.000000,0.500000,13.023980
2,0.000000,0.000000,1.000000,11.990678
3,0.500000,0.000000,0.000000,11.688487
4,0.500000,0.000000,0.500000,12.974571
...,...,...,...,...
23,0.195706,0.141341,0.000000,15.157671
24,0.000000,0.359837,0.125651,15.739101
25,0.093065,0.000000,0.000000,14.417391
26,0.000000,0.227926,0.216711,13.718295


In [5]:
# h = 0
# h = 100
# h = 90
# h = 85
# h = 83
# h = 81 # Investigation shows a failure to generate a gaussian process above 304 point dataset...
# h = 80

In [6]:
X = df[["s1","s2","b1"]].to_numpy()
# print(np.shape(X))
# X = X[0:len(X)-h]
# print(np.shape(X))

In [7]:
y = df["DeltaPolymerMass_pct"].to_numpy()
# print(np.shape(y))
# y = y[0:len(X)-h]
# print(np.shape(y))

# Data Visualisation

In [8]:
fig = px.scatter_3d(df, x='s1', y='s2', z='b1', color='DeltaPolymerMass_pct',width=1300, height=600)
fig.show()

# Model Setup

In [9]:
client = Client()

In [10]:
parameters = [
    RangeParameterConfig(
        name="s1", parameter_type="float", bounds=(0, 1)
    ),
    RangeParameterConfig(
        name="s2", parameter_type="float", bounds=(0, 1)
    ),
    RangeParameterConfig(
        name="b1", parameter_type="float", bounds=(0, 1)
    ),
]

client.configure_experiment(parameters=parameters)

In [11]:
def construct_generation_strategy(
    generator_spec: GeneratorSpec, node_name: str,
) -> GenerationStrategy:
    """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
    using the provided `generator_spec` for the Modular BoTorch node.
    """
    botorch_node = GenerationNode(
        node_name=node_name,
        model_specs=[generator_spec],
    )
    return GenerationStrategy(
        name=f"{node_name}",
        nodes=[botorch_node]
    )

construct_generation_strategy(
    generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
    node_name="Modular BoTorch",
)

GenerationStrategy(name='Modular BoTorch', nodes=[GenerationNode(node_name='Modular BoTorch', model_specs=[GeneratorSpec(model_enum=BoTorch, model_key_override=None)], transition_criteria=[])])

In [12]:
surrogate_spec = SurrogateSpec(
    model_configs=[
        ModelConfig(
            botorch_model_class=SingleTaskGP,

            # covar_module_class=MaternKernel,
            # covar_module_options={"nu": 2.5},

            covar_module_class=MaternKernel,
            covar_module_options={"nu": 1.5},

            # covar_module_class=MaternKernel,
            # covar_module_options={"nu": 0.5},

            # covar_module_class=RBFKernel,
        ),
    ],
    eval_criterion=MSE,
    allow_batched_models=False,
)

In [13]:
generator_spec = GeneratorSpec(
    model_enum=Generators.BOTORCH_MODULAR,
    model_kwargs={
        "surrogate_spec": surrogate_spec,
        "botorch_acqf_class": qLogNoisyExpectedImprovement,
        "acquisition_options": {},
    },
    model_gen_kwargs = {
        "model_gen_options": {
            "optimizer_kwargs": {
                "num_restarts": 20,
                "sequential": False,
                "options": {
                    "batch_limit": 5,
                    "maxiter": 100,
                },
            },
        },
    }
)

In [14]:
generation_strategy = construct_generation_strategy(
    generator_spec=generator_spec,
    node_name="BoTorch w/ Model Selection",
)
client.set_generation_strategy(
    generation_strategy=generation_strategy,
)

In [15]:
metric_name = "t1"
objective = f"{metric_name}"

client.configure_optimization(objective=objective)

In [16]:
for array,target in zip(X,y):
    my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
    trial_index = client.attach_trial(parameters=my_parameters)
    client.complete_trial(trial_index=trial_index,raw_data={"t1": target})

In [17]:
client.summarize()

,trial_index,arm_name,trial_status,t1,s1,s2,b1
0,0,0_0,COMPLETED,14.688580,0.000000,0.000000,0.000000
1,1,1_0,COMPLETED,13.023980,0.000000,0.000000,0.500000
2,2,2_0,COMPLETED,11.990678,0.000000,0.000000,1.000000
3,3,3_0,COMPLETED,11.688487,0.500000,0.000000,0.000000
4,4,4_0,COMPLETED,12.974571,0.500000,0.000000,0.500000
...,...,...,...,...,...,...,...
296,296,296_0,COMPLETED,15.157671,0.195706,0.141341,0.000000
297,297,297_0,COMPLETED,15.739101,0.000000,0.359837,0.125651
298,298,298_0,COMPLETED,14.417391,0.093065,0.000000,0.000000
299,299,299_0,COMPLETED,13.718295,0.000000,0.227926,0.216711


# Digging Around

In [ ]:
client.get_next_trials(max_trials=1)

In [ ]:
client.predict([{"s1":0.1,"s2":0.1,"b1":0.1}])[0]["t1"][0]

In [ ]:
n = 10
iCoords_arr = np.linspace(0,1,n-1)
jCoords_arr = np.linspace(0,1,n-1)
kCoords_arr = np.linspace(0,1,n-1)
ijkCoordsOld_lis = []
ijkCoords_lis = []
for i in iCoords_arr:
    for j in jCoords_arr:
        for k in kCoords_arr:
            ijkCoordsOld_lis.append([i,j,k])
            ijkCoords_lis.append({"s1":i,"s2":j,"b1":k})
y_pred = client.predict(ijkCoords_lis)
y_pred_lis = []
for i in y_pred:
    y_pred_lis.append(i["t1"][0])
ijkCoordsOld_arr = np.array(ijkCoordsOld_lis)
y_pred_arr = np.array(y_pred_lis)
df2 = pd.DataFrame({'s1': ijkCoordsOld_arr[:, 0],'s2': ijkCoordsOld_arr[:, 1],'b1': ijkCoordsOld_arr[:, 2], 'y_pred': y_pred_arr})

fig = px.scatter_3d(df2, x='s1', y='s2', z='b1', color='y_pred')
fig.show()

print(np.max(y_pred_arr))
print(ijkCoordsOld_lis[np.argmax(y_pred_arr)])
print(df2)

# Saving

In [ ]:
client.save_to_json_file('ModelMk6.json')

# lOADING

In [ ]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
client.get_next_trials(max_trials=1)
n = 100
iCoords_arr = np.linspace(0,1,n-1)
jCoords_arr = np.linspace(0,1,n-1)
kCoords_arr = np.linspace(0,1,n-1)
ijkCoordsOld_lis = []
ijkCoords_lis = []
for i in iCoords_arr:
    for j in jCoords_arr:
        for k in kCoords_arr:
            ijkCoordsOld_lis.append([i,j,k])
            ijkCoords_lis.append({"s1":i,"s2":j,"b1":k})
y_pred = client.predict(ijkCoords_lis)
y_pred_lis = []
for i in y_pred:
    y_pred_lis.append(i["t1"][0])
ijkCoordsOld_arr = np.array(ijkCoordsOld_lis)
y_pred_arr = np.array(y_pred_lis)
df2 = pd.DataFrame({'s1': ijkCoordsOld_arr[:, 0],'s2': ijkCoordsOld_arr[:, 1],'b1': ijkCoordsOld_arr[:, 2], 'y_pred': y_pred_arr})

fig = px.scatter_3d(df2, x='s1', y='s2', z='b1', color='y_pred')
fig.show()

print(np.max(y_pred_arr))
print(ijkCoordsOld_lis[np.argmax(y_pred_arr)])
print(df2)